# E6 - Associative recall: bo loc ngan co lam mat kha nang tam xa khong?

Repo: https://github.com/KienNguyenDev2711/Hyena-Attention-Study

## Hai lan chay truoc deu HONG - va deu do loi thiet ke, khong phai do mo hinh

**Lan 1 (tran bao hoa).** Dong chon cau hinh viet `solvable[-1]` trong khi comment ghi "cau hinh kho nhat"; phan tu cuoi lai la cau hinh DE NHAT (vocab=5, L=17). Ca bon cau hinh deu dat ~1,000, chenh lech 0,0005 = 1 mau tren 2000 = nhieu.

**Lan 2 (moi thu ve muc doan mo).** Hai loi chong nhau:

| Loi | Bang chung |
|---|---|
| Test R5 lam roi mat lich **warmup** | cung 18720 buoc, cung lr=1e-3: **co** warmup -> 1,000 · **khong** warmup -> 0,187 |
| Quet do dai dat `vocab = n_pairs` | L tu 33->257 lam TANG DONG THOI khoang cach, so cap phai nho, va so lop dau ra (doan mo tut 6,2% -> 0,8%). Khong quy duoc ket qua cho truc nao. |

## Thiet ke lan nay: co lap DUNG MOT bien

Dung `n_pairs_fixed` de giu nguyen phep tra cuu (cung so cap, cung vocab) va keo dai chuoi bang **token dem**:

```
[k1 v1 k2 v2 ... kP vP] [dem dem ... dem] [truy van]
 <---- 2P token ---->    <---- F ---->     1 token
```

Token dem lay ngau nhien tu chinh bang chu cai nen mo hinh khong nhan ra chung bang mot ky hieu rieng - no buoc phai mang thong tin qua ca doan dem. Da kiem chung (test R4d): chen them F token thi khoang cach truy van tang dung F.

Ba pha, moi pha co chot chan de khong dot GPU vo ich:
0. **Kiem chung cong thuc** - warmup co that su quyet dinh khong? 2 lan chay, ~3 phut.
1. **Do tham** - chi attention, tim khoang cach xa nhat con giai duoc.
2. **So sanh** - 4 cau hinh, chi o khoang cach do.

Settings: **Accelerator = GPU T4 x2**, **Internet = On**.

In [ ]:
REPO_URL = "https://github.com/KienNguyenDev2711/Hyena-Attention-Study.git"
WORK = "/kaggle/working/Hyena-Attention-Study"

import os, shutil, subprocess, sys

if os.path.isdir(WORK):
    shutil.rmtree(WORK)
subprocess.run(["git", "clone", "--depth", "1", "-q", REPO_URL, WORK], check=True)
os.chdir(WORK); sys.path.insert(0, WORK)

import torch
print("torch", torch.__version__)
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} - {p.total_memory/2**30:.1f} GB")
else:
    print("\n" + "!" * 70)
    print("!! CHUA BAT GPU - Settings > Accelerator > GPU T4 x2")
    print("!" * 70)

In [ ]:
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from hyena_study.data.synthetic import (
    RecallConfig, build_recall_dataset, chance_accuracy, query_distance)
from hyena_study.models import HyenaFilterConfig, LMConfig, SequenceLM

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Cau hinh tra cuu GIU CO DINH o moi lan chay. Day la cau hinh da duoc chung
# minh giai duoc (attention dat 1,000 voi 18720 buoc + warmup).
VOCAB, N_PAIRS, D_MODEL = 10, 16, 64
STEPS_TARGET, N_TRAIN, BS, LR = 18720, 20000, 64, 1e-3


def run_one(n_filler, layers, alpha=None, seed=0, warmup_frac=0.05):
    """Huan luyen mot cau hinh. Chi `n_filler` thay doi giua cac lan quet."""
    torch.manual_seed(seed); np.random.seed(seed)
    L = 2 * N_PAIRS + n_filler + 1
    cfg = RecallConfig(vocab_size=VOCAB, seq_len=L, n_pairs_fixed=N_PAIRS,
                       n_train=N_TRAIN, n_val=10, n_test=2000, seed=0)
    d = build_recall_dataset(cfg)
    xtr, ytr = d["train"]; xte, yte = d["test"]

    m = SequenceLM(LMConfig(
        vocab_size=VOCAB, d_model=D_MODEL, layer_spec=layers, max_seq_len=L,
        dropout=0.0, n_heads=4,
        hyena_filter=HyenaFilterConfig(alpha_values=alpha),
    )).to(DEV)
    opt = torch.optim.AdamW(m.parameters(), lr=LR)

    xtr_t = torch.from_numpy(xtr).to(DEV); ytr_t = torch.from_numpy(ytr).to(DEV)
    per_epoch = len(xtr) // BS
    epochs = max(1, round(STEPS_TARGET / per_epoch))
    total = epochs * per_epoch
    warmup = max(int(total * warmup_frac), 1)
    step, t0 = 0, time.time()

    for _ in range(epochs):
        perm = torch.randperm(len(xtr), device=DEV)
        for i in range(0, len(xtr) - BS + 1, BS):
            idx = perm[i:i + BS]
            for g in opt.param_groups:
                g["lr"] = LR * min(1.0, (step + 1) / warmup)
            loss = F.cross_entropy(m(xtr_t[idx])[:, -1, :], ytr_t[idx])
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
            step += 1

    m.eval()
    with torch.no_grad():
        pred = m(torch.from_numpy(xte).to(DEV))[:, -1, :].argmax(-1).cpu().numpy()
    acc = float((pred == yte).mean())

    dist = query_distance(xte, n_pairs=N_PAIRS)
    half = np.median(dist)
    near = float((pred[dist <= half] == yte[dist <= half]).mean())
    far = float((pred[dist > half] == yte[dist > half]).mean())

    return {"n_filler": n_filler, "seq_len": L, "accuracy": acc,
            "chance": chance_accuracy(cfg), "acc_near": near, "acc_far": far,
            "dist_mean": float(dist.mean()), "dist_max": int(dist.max()),
            "steps": step, "seconds": time.time() - t0}


def solved(r):
    return r["accuracy"] > r["chance"] + 0.30


print(f"Tra cuu co dinh: vocab={VOCAB}, {N_PAIRS} cap, doan mo={1/VOCAB:.3f}")
print(f"Ngan sach: {STEPS_TARGET} buoc, lr={LR}")

## Pha 0 - Warmup co that su la nguyen nhan khong?

Chay dung mot cau hinh, chi khac nhau o lich warmup. Neu gia thuyet dung: co warmup ~1,000, khong warmup ~0,19.

Neu **ca hai** deu that bai thi gia thuyet cua toi sai va phai dung lai de chan doan tiep - dung chay pha 1 va 2.

In [ ]:
print(f"{'warmup':>10}{'acc':>9}{'doan mo':>10}{'giay':>7}")
print("-" * 36)
phase0 = []
for wf, tag in ((0.05, "co (5%)"), (0.0, "khong")):
    r = run_one(n_filler=0, layers="AA", warmup_frac=wf if wf > 0 else 1e-9)
    r["warmup"] = tag
    phase0.append(r)
    print(f"{tag:>10}{r['accuracy']:>9.3f}{r['chance']:>10.3f}{r['seconds']:>7.0f}")

pd.DataFrame(phase0).to_csv("results/E6_phase0_warmup.csv", index=False)

RECIPE_OK = solved(phase0[0])
print()
if RECIPE_OK and not solved(phase0[1]):
    print("=> Gia thuyet DUNG: warmup la yeu to quyet dinh. Chay tiep pha 1.")
elif RECIPE_OK:
    print("=> Ca hai deu giai duoc: warmup KHONG phai nguyen nhan that su cua")
    print("   lan that bai truoc. Van chay tiep duoc, nhung phai tim lai nguyen")
    print("   nhan that su truoc khi viet vao bao cao.")
else:
    print("=> CA HAI DEU THAT BAI. Gia thuyet sai. DUNG LAI, dung chay pha 1-2.")
    print("   Nghi ngo tiep theo: chenh lech giua ham trial() cu va run_one() moi")
    print("   (kich thuoc tap val, cach xao tron, thu tu sinh du lieu).")

## Pha 1 - Do tham: attention chiu duoc khoang cach bao xa?

Chi tang `n_filler`. Phep tra cuu giu nguyen do kho, chi khoang cach thay doi. Dung ngay khi mot muc that bai (xa hon chac chan cung that bai).

In [ ]:
FILLERS = [0, 32, 96, 224, 480]

scout, TARGET = [], None
if not RECIPE_OK:
    print("Bo qua pha 1 vi pha 0 chua dat.")
else:
    print(f"{'dem':>6}{'L':>6}{'k/cach TB':>11}{'acc':>8}{'gan':>7}{'xa':>7}{'giay':>7}")
    print("-" * 52)
    for f in FILLERS:
        r = run_one(n_filler=f, layers="AA")
        r["config"] = "attention"; r["solvable"] = solved(r)
        scout.append(r)
        print(f"{f:>6}{r['seq_len']:>6}{r['dist_mean']:>11.1f}{r['accuracy']:>8.3f}"
              f"{r['acc_near']:>7.3f}{r['acc_far']:>7.3f}{r['seconds']:>7.0f}"
              + ("  giai duoc" if r["solvable"] else "  chua dat"))
        if not r["solvable"]:
            print("     -> dung quet, khoang cach xa hon chac chan cung that bai")
            break

    pd.DataFrame(scout).to_csv("results/E6_scout_attention.csv", index=False)
    ok = [r for r in scout if r["solvable"]]
    if ok:
        TARGET = max(ok, key=lambda r: r["n_filler"])
        print(f"\n=> Pha 2 dung n_filler={TARGET['n_filler']} (L={TARGET['seq_len']}, "
              f"khoang cach TB {TARGET['dist_mean']:.0f}, attention {TARGET['accuracy']:.3f})")
        if TARGET["accuracy"] > 0.995:
            print("   CANH BAO: attention gan nhu tuyet doi o muc nay. Neu ca 4 cau")
            print("   hinh cung bao hoa thi lai khong ket luan duoc - can tang FILLERS.")
    else:
        print("\n=> Khong muc nao giai duoc. Bo qua pha 2.")

## Pha 2 - So sanh bon cau hinh

| Cau hinh | Y nghia |
|---|---|
| `attention` | moc tren, truy cap truc tiep moi vi tri |
| `hyena_uniform` | khoang alpha nhom tu chon (doi chung 1) |
| `hyena_logspace` | do dai hieu dung trai deu theo log (doi chung 2, cong bang) |
| `hyena_corpus` | alpha suy tu corpus (de xuat), trung vi ~3 token |

**Du doan can kiem chung:** `corpus` co do dai hieu dung rat ngan nen se kem nhat, ro nhat o cot `acc_far`.

In [ ]:
from hyena_study.morphology import alphas_from_mi, logspaced_alphas

if TARGET is None:
    print("Bo qua pha 2: pha truoc chua dat.")
else:
    L = TARGET["seq_len"]
    # Alpha phai SINH LAI cho dung d_model va do dai cua tac vu nay; dung lai
    # file alpha sinh cho d_model=256 / L=512 se lam sai do dai hieu dung.
    mi = pd.read_csv("results/E0b_mi_decay_vi_bpe_k500.csv")
    corpus_alpha = alphas_from_mi(mi["lag"].values, mi["mi_corrected_nats"].values,
                                  d_model=D_MODEL, seq_len=L).alpha
    logspace_alpha = logspaced_alphas(D_MODEL, seq_len=L).alpha
    for nm, a in (("corpus", corpus_alpha), ("logspace", logspace_alpha)):
        e = L / np.array(a)
        print(f"  {nm:<9} do dai hieu dung: trung vi {np.median(e):.1f}, "
              f"{(e <= 4).sum()}/{D_MODEL} kenh <= 4 token, xa nhat {e.max():.0f}")

    CONFIGS = [("attention", "AA", None), ("hyena_uniform", "HH", None),
               ("hyena_logspace", "HH", logspace_alpha),
               ("hyena_corpus", "HH", corpus_alpha)]
    rows = []
    print(f"\n{'cau hinh':<16}{'seed':>5}{'acc':>8}{'gan':>7}{'xa':>7}{'giay':>7}")
    print("-" * 50)
    for name, spec, alpha in CONFIGS:
        for sd in (0, 1):
            r = run_one(TARGET["n_filler"], spec, alpha=alpha, seed=sd)
            r["config"] = name; r["seed"] = sd
            rows.append(r)
            print(f"{name:<16}{sd:>5}{r['accuracy']:>8.4f}{r['acc_near']:>7.3f}"
                  f"{r['acc_far']:>7.3f}{r['seconds']:>7.0f}")

    df = pd.DataFrame(rows)
    df.to_csv("results/E6_recall_comparison.csv", index=False)
    print("\n" + "=" * 66)
    print(df.groupby("config")[["accuracy", "acc_near", "acc_far"]]
            .agg(["mean", "std"]).to_string())
    print(f"\nDoan mo {rows[0]['chance']:.3f} | L={rows[0]['seq_len']} | "
          f"khoang cach TB {rows[0]['dist_mean']:.0f}")
    print("\nCACH DOC:")
    print("  * Moi cau hinh deu ~1,000 => lai bao hoa, KHONG ket luan gi,")
    print("    phai tang FILLERS roi chay lai.")
    print("  * corpus co acc_far thap hon logspace => khoi tao tu du lieu danh")
    print("    doi kha nang tam xa. Day la phat hien can tim.")
    print("  * Chenh lech nho hon do lech giua hai seed => nhieu, khong ket luan.")

In [ ]:
import shutil
from pathlib import Path

out = Path("/kaggle/working/E6_ketqua")
if out.exists():
    shutil.rmtree(out)
out.mkdir(parents=True)
for f in Path("results").glob("E6*"):
    shutil.copy(f, out / f.name)
shutil.make_archive("/kaggle/working/E6_ketqua", "zip", out)
print("Da gom:", sorted(p.name for p in out.iterdir()))
print("Tai /kaggle/working/E6_ketqua.zip o panel Output ben phai.")